In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

print(device)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *


cpu


In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        # для инварианта положительности, пока нет другого фикса
        self.fc_input = nn.Linear(input_size, hidden_size)
        self.act = nn.ReLU()
        self.fc_output = nn.Linear(hidden_size, 10)


    def forward(self, x):
        x = self.fc_input(x)
        x = self.act(x)
        x = self.fc_output(x)
        return x


In [4]:
hyperparams = {
    "num_epochs": 50,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc_output": 0.4},
    "pruning_thresholds": {"fc_output": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 1e-4,
    "diff_threshold": 0.05,
    "warmup_epochs": 5
}

In [5]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [6]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)

Converting layer 'fc_output' to ScaledNet...


In [7]:
# import wandb
# wandb.login()

In [8]:
# wandb.finish()
# run = wandb.init(
#     project="self-expanding-nets",
#     name=f"name",
#     config=hyperparams
# )

In [9]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="mnist",
)
experiment.log_parameters(hyperparams)
experiment.set_name("mnist origin")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gcof/mnist/ce3f5f6ebe7d4177ad69726d054f466f



In [10]:
criterion = nn.CrossEntropyLoss()

optimizer_cls = partial(
    optim.Adam,
    lr=hyperparams['learning_rate'],
    weight_decay=hyperparams['weight_decay']
)

trained_model = train_ssonn(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    finder_loader=finder_loader,
    criterion=criterion,
    optimizer_cls=optimizer_cls,
    hyperparams=hyperparams,
    device=device,
    experiment=experiment,
    use_log_tool=True,
)


--- Starting SSONN Training ---


Train:   0%|          | 0/188 [00:00<?, ?it/s]COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


Epoch 1/50 | T-Loss: 1.0321 | V-Loss: 0.4806 | Acc: 0.8796 | LRs: [1.0e-03] | Params: 12730/12730


Epoch 2/50 | T-Loss: 0.3963 | V-Loss: 0.3493 | Acc: 0.9059 | LRs: [1.0e-03] | Params: 12730/12730


Epoch 3/50 | T-Loss: 0.3246 | V-Loss: 0.3110 | Acc: 0.9139 | LRs: [1.0e-03] | Params: 12730/12730


Epoch 4/50 | T-Loss: 0.2951 | V-Loss: 0.2921 | Acc: 0.9196 | LRs: [1.0e-03] | Params: 12730/12730


... (42 outputs left out)

Epoch 47/50 | T-Loss: 0.1265 | V-Loss: 0.1694 | Acc: 0.9530 | LRs: [5.0e-04 | 5.0e-04 | 1.0e-03 | 2.0e-04] | Params: 49719/50189


Epoch 48/50 | T-Loss: 0.1204 | V-Loss: 0.1682 | Acc: 0.9516 | LRs: [5.0e-04 | 5.0e-04 | 1.0e-03 | 4.0e-04] | Params: 49872/50189


Epoch 49/50 | T-Loss: 0.1173 | V-Loss: 0.1618 | Acc: 0.9538 | LRs: [5.0e-04 | 5.0e-04 | 1.0e-03 | 6.0e-04] | Params: 49872/50189


Epoch 50/50 | T-Loss: 0.1130 | V-Loss: 0.1616 | Acc: 0.9523 | LRs: [5.0e-04 | 5.0e-04 | 1.0e-03 | 8.0e-04] | Params: 49872/50189
--- Training Finished ---


In [11]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mnist origin
COMET INFO:     url                   : https://www.comet.com/gcof/mnist/ce3f5f6ebe7d4177ad69726d054f466f
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     expansion event [3] : (28, 153)
COMET INFO:     lr [50]             : (0.0005, 0.001)
COMET INFO:     lr_group_0 [50]     : (0.0005, 0.001)
COMET INFO:     lr_group_1 [32]     : (0.0, 0.001)
COMET INFO:     lr_group_2 [14]     : (0.0, 0.001)
COMET INFO:     lr_group_3 [5]      : (0.0, 0.0008)
COMET INFO:     params amount [50]  : (12730, 49872)
COMET INFO:     sparsity [50]       : (0.0, 0.014308881045502209)
COMET INFO:     total params [50]   : (12730, 50189)
COMET INFO:    

In [43]:
torch.save(model, 'mnist.pt')